In [14]:
import pandas as pd
from pathlib import Path

pois_path = Path("Areas-of-interest-POIs") / "Data-created(2nd LLM run)" / "filtered_mid_label_notna_bosserhof_na.parquet"

pois = pd.read_parquet(pois_path)

print(f"Loaded {len(pois)} rows")

Loaded 68 rows


In [15]:
pois.head()

,gml_id,osm_names,volume_m3,sentence,interpreted_type,bosserhof_class,mid_label,geometry
0,10752,None,143.856934,general_building_context: building_label=resid...,allotment garden hut,None,[leisure],b'\x01\xee\x03\x00\x00\x01\x00\x00\x00\x01\xeb...
1,10748,None,32.816721,general_building_context: building_label=resid...,allotment garden hut,None,[leisure],b'\x01\xee\x03\x00\x00\x01\x00\x00\x00\x01\xeb...
2,10758,None,47.426468,general_building_context: building_label=resid...,allotment garden hut,None,[leisure],b'\x01\xee\x03\x00\x00\x01\x00\x00\x00\x01\xeb...
3,10760,None,86.428497,general_building_context: building_label=resid...,allotment garden hut,None,[leisure],b'\x01\xee\x03\x00\x00\x01\x00\x00\x00\x01\xeb...
4,14286,None,30.406003,general_building_context: building_label=resid...,allotment garden house,None,[leisure],b'\x01\xee\x03\x00\x00\x01\x00\x00\x00\x01\xeb...


In [16]:
pois = pois.drop(columns=['sentence','interpreted_type','bosserhof_class', 'mid_label', 'geometry'],axis=1)

In [17]:
pois.columns

Index(['gml_id', 'osm_names', 'volume_m3'], dtype='object')

In [18]:
import pandas as pd

def is_missing(x) -> bool:
    if x is None:
        return True
    if isinstance(x, float) and pd.isna(x):
        return True
    if isinstance(x, str) and x.strip() == "":
        return True
    if isinstance(x, (list, tuple, set, dict)) and len(x) == 0:
        return True
    return False


def format_value(x):
    if is_missing(x):
        return None

    if isinstance(x, (list, tuple, set)):
        vals = [str(v).strip() for v in x if not is_missing(v)]
        vals = [v for v in vals if v != ""]
        return "; ".join(vals) if vals else None

    if isinstance(x, dict):
        vals = []
        for k, v in x.items():
            if not is_missing(v):
                vals.append(f"{k}: {v}")
        return "; ".join(vals) if vals else None

    return str(x).strip()


def row_to_sentence(row) -> str:
    parts = []
    parts.append("This row describes one building.")

    # Name / identity
    identity_bits = []
    if format_value(row.get("osm_names")):
        identity_bits.append(f"Known OSM name(s): {format_value(row.get('osm_names'))}")
    if format_value(row.get("label_en")):
        identity_bits.append(f"Building label: {format_value(row.get('label_en'))}")
    if format_value(row.get("gfk_class")):
        identity_bits.append(f"GFK class: {format_value(row.get('gfk_class'))}")

    if identity_bits:
        parts.append("Building identity: " + ". ".join(identity_bits) + ".")

    # Address
    if format_value(row.get("alkis_address")):
        parts.append(f"Address from ALKIS: {format_value(row.get('alkis_address'))}.")

    # OSM typed attributes
    osm_attr_bits = []
    for col, label in [
        ("amenity", "amenity"),
        ("building", "building"),
        ("shop", "shop"),
        ("tourism", "tourism"),
        ("information", "information"),
        ("osm_building_type", "OSM building type"),
        ("osm_landuse_class", "OSM land-use class"),
        ("osm_landuse_name", "OSM land-use name"),
    ]:
        val = format_value(row.get(col))
        if val:
            osm_attr_bits.append(f"{label}: {val}")

    if osm_attr_bits:
        parts.append("OSM-derived attributes: " + "; ".join(osm_attr_bits) + ".")

    # Other official / building-purpose clues
    official_bits = []
    if format_value(row.get("ALKIS_Landuse_info")):
        official_bits.append(f"ALKIS land-use information: {format_value(row.get('ALKIS_Landuse_info'))}")

    if official_bits:
        parts.append("Official land-use clues: " + "; ".join(official_bits) + ".")

    # Contact / online presence
    contact_bits = []
    if format_value(row.get("email")):
        contact_bits.append(f"email: {format_value(row.get('email'))}")
    if format_value(row.get("website")):
        contact_bits.append(f"website: {format_value(row.get('website'))}")

    if contact_bits:
        parts.append("Contact or web clues: " + "; ".join(contact_bits) + ".")

    # Free text
    if format_value(row.get("additional_information")):
        parts.append(f"Additional descriptive information: {format_value(row.get('additional_information'))}.")
    if format_value(row.get("tags_search")):
        parts.append(f"Search tags or keywords: {format_value(row.get('tags_search'))}.")

    # Geometry / size clue
    if not is_missing(row.get("volume_m3")):
        parts.append(f"Estimated building volume: {row.get('volume_m3')} cubic meters.")

    return " ".join(parts).strip()


pois["sentence"] = pois.apply(row_to_sentence, axis=1)

In [19]:
pois.head()

,gml_id,osm_names,volume_m3,sentence
0,10752,None,143.856934,This row describes one building. Estimated bui...
1,10748,None,32.816721,This row describes one building. Estimated bui...
2,10758,None,47.426468,This row describes one building. Estimated bui...
3,10760,None,86.428497,This row describes one building. Estimated bui...
4,14286,None,30.406003,This row describes one building. Estimated bui...


In [20]:
import json
import time
import re
import requests
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_PATH = Path("Areas-of-interest-POIs") / ".env"
load_dotenv(dotenv_path=ENV_PATH)

TU_TOKEN = os.getenv("TU_KI_TOOLBOX_TOKEN")

if not TU_TOKEN:
    raise RuntimeError(
        "Missing TU_KI_TOOLBOX_TOKEN."
        "Check Areas-of-interest-POIs/.env"
    )

API_URL = "https://ki-toolbox.tu-braunschweig.de/api/v1/chat/send"
# MODEL = "gpt-5.4-2026-03-05" # chatgpt-5.2
MODEL = "gpt-oss-120b" # open source chatgpt-oss-120b

In [21]:
SYSTEM_PROMPT = """
You are a building activity interpreter and classifier.

Your job is to infer what a place most likely is from a text snippet
and classify it in TWO WAYS:
1) Activity labels (visitor intent / building use based on the label set)
2) A Bosserhof-based building-use class for capacity / volume estimation

IMPORTANT SCOPE RESTRICTION (CRITICAL)
This task is about CLASSIFYING BUILDINGS OR BUILDING-LIKE PLACES ONLY.

- If the described place is NOT a building or not a place people enter/use as a destination,
  you MUST return:
    - "mid_labels": []
    - "bosserhof_class": null
- Do NOT assign any class to pure outdoor objects, infrastructure, or non-living POIs.

INPUT
You will receive ONE text describing a place. It may contain:
- Name, address, city
- OSM tags: amenity=*, shop=*, office=*, leisure=*, tourism=*, building=*, landuse=*
- Free-form keywords (German/English): “Gymnasium”, “Kita”, “Praxis”, “Rathaus”, “Universität”, etc.

────────────────────────────────────
PART A — ACTIVITY LABELS (VISITOR INTENT)
────────────────────────────────────

ALLOWED ACTIVITY LABELS
- work
- university
- school
- childcare
- retail_daily
- retail_non_daily
- leisure
- sports
- errands
- meetup
- lessons
- business

CORE PRINCIPLE
Classify by VISITOR INTENT to a BUILDING:
What do people primarily go there FOR?

MULTI-LABEL RULE (CRITICAL)
A building may support MULTIPLE activities.
Assign ALL labels that clearly apply to the building’s typical use.
Do not force labels that are only weakly implied.

────────────────────────────────────
LABEL DEFINITIONS
────────────────────────────────────

- work

Represents on-site labor, operational activity, or institutional functioning carried out by people inside the building.
Assign this label whenever the building’s normal use depends on people being physically present
to carry out tasks, responsibilities, operations, supervision, coordination, production,
maintenance, administration, instruction, care, or service provision.
The focus of this label is not why visitors come, but whether the building functions as a place
where human work is performed as part of its regular purpose.
A building should receive "work" whenever people are not merely present incidentally,
but are there in an active functional role that helps the building fulfill its purpose.
This includes cases where the same building also serves customers, students, patients, clients,
guests, or other visitors. In such cases, "work" must be added in parallel with the visitor-oriented label(s),
because the activity of workers is part of the building’s core function.
Do not require explicit evidence of employment words or job titles.
Infer "work" from the meaning of the building, the type of activity taking place there,
and the fact that its function normally depends on people performing organized tasks on-site.
Do NOT assign "work" when human presence is only private, incidental, passive, or non-functional,
such as merely being at home, visiting socially, resting, or informally occupying the space
without contributing to the building’s operation or organized purpose.
Core idea: "People are here not only to use the building, but also to make it function."

- university

Represents tertiary/higher-level education activity.
Covers structured learning, teaching, and research at the higher education level
Involves academic instruction, research, and study environments
Distinct from general learning by its institutional and advanced nature
Core idea: “Advanced academic education and research happen here.”

- school

Represents formal compulsory or pre-tertiary education activity.
Covers structured education for children and adolescents
Includes general and vocational schooling
Defined by curriculum-based learning under institutional supervision
Core idea: “Children or teenagers receive structured education here.”

- childcare

Represents supervision and early development care for young children.
Focuses on care, supervision, and early-stage development
Not primarily academic or curriculum-driven (unlike school)
Strong emphasis on custodial and developmental support
Core idea: “Young children are cared for and supervised here.”

- retail_daily

Represents frequent, necessity-driven consumption activity.
Covers acquisition of essential, regularly needed goods
Characterized by high frequency and routine visits
Typically tied to basic living needs
Core idea: “People come here regularly to fulfill essential daily needs.”

- retail_non_daily

Represents infrequent, discretionary consumption activity.
Covers acquisition of non-essential or durable goods
Visits are planned, occasional, or need-based
Often involves comparison, browsing, or larger purchases
Core idea: “People come here occasionally to buy non-essential or long-term goods.”

- leisure

Represents hedonic, relaxation, or enjoyment-oriented activity.
Focuses on free-time use driven by pleasure, comfort, or experience
Includes passive or active enjoyment, but not primarily goal-oriented tasks
Distinguished by non-obligatory participation. People choose to go here for enjoyment, not out of necessity or obligation.
Core idea: “People come here to relax, enjoy, or spend free time.”

- sports

Represents physical activity, exercise, or bodily training.
Focuses on intentional physical exertion or fitness improvement
Can be recreational or structured, but always movement-centered
Distinguished from leisure by physical intensity and purpose
Core idea: “People come here to be physically active or train their body.”

- errands

Represents task-oriented, functional activities needed for daily life management.
Covers practical obligations or maintenance tasks
Typically short, goal-driven visits with a clear outcome
Often involves services, administration, or personal maintenance
Core idea: “People come here to complete necessary tasks or obligations.”

- meetup

Represents social interaction and gathering activity.
Focuses on interpersonal connection and shared presence
Can be informal or organized, but the primary purpose is social exchange
Not necessarily tied to consumption or formal structure
Core idea: “People come here to meet and interact with others.”

- lessons

Represents structured instruction or skill acquisition activity.
Focuses on learning guided by an instructor
Can occur at any level (formal or informal), but always organized and instructional
Distinguished from general education by activity type (learning session), not institution
Core idea: “People come here to learn something through instruction.”

- business

Represents professional or organizational interaction activity (non-routine workplace).
Covers goal-oriented professional interactions, often external-facing
Includes meetings, consulting, administrative dealings, or formal exchanges
Distinct from "work" because it reflects the visitor’s purpose, not employment
Core idea: “People come here for professional or organizational matters.”

IMPORTANT
- Multiple labels are allowed and encouraged when clearly supported
- Prefer precision over over-labeling
- If there is no clear building-related visitor activity, return "mid_labels": []

────────────────────────────────────
PART B — BOSSERHOF BUILDING-USE CLASS
────────────────────────────────────

GOAL
Assign EXACTLY ONE Bosserhof label that matches the categories/subcategories.

This label is for capacity / floor-area / employee-intensity estimation,
so it MUST reflect the dominant FUNCTIONAL USE OF THE BUILDING.

STEP 1 — Subcategory FIRST (preferred)
If you can confidently map the building to ONE of the Bosserhof SUBCATEGORIES listed below,
output that subcategory label (exact string match).

STEP 2 — Headline category fallback (MANDATORY WHEN SUBCATEGORY IS UNCERTAIN)
If the exact subcategory is uncertain but the dominant functional sector is inferable,
you MUST output the corresponding HEADLINE category.
Do not return null when a headline category can reasonably represent the building's dominant use.

STEP 3 — No assignment
If the building does not fit reliably in any headline category OR is not an enterable building,
output bosserhof_class = null.

────────────────────────────────────
ALLOWED BOSSERHOF HEADLINE CATEGORIES AND SUBCATEGORIES
────────────────────────────────────

NEAREST-VALID-CLASS RULE (MANDATORY)

If the building is clearly enterable and supports a recognizable human activity,
you MUST assign the closest Bosserhof class, even if the fit is imperfect.

1) Transport
- no fixed subcategories given; use headline when transport building with staff is clear
Notes: Transport-related buildings with operational staff (depots, terminals)
Exclude: stops/platforms/tracks

2) Yards, depots, storage areas, construction yards
- no fixed subcategories given; use headline when storage/yard with staff is clear
Notes: Storage or operational yards with staff
Exclude: pure storage without staff, open yards without buildings

3) Industrial operations / Production
Subcategories:
- highly productive industries / machine / material or space intensive
- others

4) Crafts and trades
Subcategories:
- craft businesses
- craft courtyards

5) Services
Subcategories:
- normal office
- open-plan office
- business-oriented services
- customer-oriented services
- hotels
- hotels with conference areas
- restaurants / gastronomy
- suppliers for car dealerships
- vehicle / electrical repair
- customer service
- car dealerships

6) Retail
Subcategories:
- wholesale
- retail (small-scale)
- discount stores
- DIY stores
- furniture stores
- hypermarkets / superstores
- shopping centers
- self-service department stores
- department stores
- factory outlet centers

7) Public facilities
Subcategories:
- schools
- universities
- research institutes
- kindergartens
- hospitals
- nursing homes

8) Facilities for culture, leisure and sports
Subcategories:
- entertainment, culture
- large cinemas
- musical theatres
- large discos, fun / leisure pools
- arenas, large events
- theme parks
- fitness / wellness

────────────────────────────────────
OUTPUT FORMAT (STRICT JSON ONLY)
────────────────────────────────────

{
  "interpreted_type": "<plain-English description of what the place most likely is>",
  "mid_labels": ["<zero or more activity labels from the allowed list>"],
  "bosserhof_class": "<one Bosserhof class or null>",
  "reason": "<max 200 words. Explain both classifications, referencing OSM tags, keywords, and any verification used. Explicitly justify the Bosserhof choice or why it is null.>"
}
""".strip()

In [22]:
# -----------------------------
# MiD labels (STRICT)
# -----------------------------

TARGET_MID_LABELS = {
    "work",
    "university",
    "school",
    "childcare",
    "retail_daily",
    "retail_non_daily",
    "leisure",
    "sports",
    "errands",
    "meetup",
    "lessons",
    "business"
}
# Felix proposed an extended set of MiD labels for more granularity and better coverage of activities. The original MiD labels are a subset of these, so the model can still assign the original ones when appropriate.

#     "work",
#     "university",
#     "school",
#     "childcare",
#     "errands",
#     "retail_non_daily",
#     "retail_daily",
#     "leisure",
# } Lasse proposed classes for Mid Labels!

# -----------------------------
# JSON extraction
# -----------------------------
def extract_first_json_object(text: str) -> dict:
    """
    Extract the first JSON object from model output.
    """
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in output.")
    return json.loads(m.group(0))


# -----------------------------
# Validation (PROMPT-ALIGNED)
# -----------------------------
def validate(obj: dict):
    # ---- Required keys ----
    if "interpreted_type" not in obj or not isinstance(obj["interpreted_type"], str):
        raise ValueError("Missing or invalid interpreted_type")

    if "mid_labels" not in obj or not isinstance(obj["mid_labels"], list):
        raise ValueError("mid_labels must be a list")

    if "bosserhof_class" not in obj:
        raise ValueError("Missing bosserhof_class")

    if "reason" not in obj or not isinstance(obj["reason"], str):
        raise ValueError("Missing or invalid reason")

    # ---- Validate MiD labels ----
    for lab in obj["mid_labels"]:
        if lab not in TARGET_MID_LABELS:
            raise ValueError(f"Invalid MiD label: {lab}")

    # ---- Validate Bosserhof class ----
    bc = obj["bosserhof_class"]

    # Allowed: no assignment
    if bc is None or bc == []:
        pass

    # Allowed: any non-empty string (granular OR fallback)
    elif isinstance(bc, str) and bc.strip():
        pass

    else:
        raise ValueError("bosserhof_class must be a non-empty string, [] or None")

    # ---- Consistency rule ----
    # If there is no building-related visitor activity,
    # there must not be a Bosserhof assignment
    if len(obj["mid_labels"]) == 0 and bc not in (None, []):
        raise ValueError(
            "bosserhof_class assigned but mid_labels is empty — inconsistent"
        )

In [23]:
def call_tu_llm(
    user_input: str,
    max_retries=3,
    backoff_sec=2.0,
    debug=False,
    reasoning_effort="high",   # "low", "medium", or "high"
) -> str:

    headers = {
        "Authorization": f"Bearer {TU_TOKEN}",
        "Accept": "application/json",
        "Content-Type": "application/json",
    }

    payload = {
        "thread": None,
        "prompt": user_input,
        "model": MODEL,
        "customInstructions": SYSTEM_PROMPT,
        "hideCustomInstructions": True,
        "reasoning": {
            "effort": reasoning_effort
        },
    }

    last_err = None

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(
                API_URL,
                headers=headers,
                json=payload,
                stream=True,
                timeout=120,
            )

            if debug:
                print("STATUS:", r.status_code)
                try:
                    print("REQUEST PAYLOAD:", json.dumps(payload, indent=2)[:2000])
                except Exception:
                    pass

            r.raise_for_status()

            full_text = ""

            for line in r.iter_lines(decode_unicode=True):
                if not line:
                    continue
                try:
                    event = json.loads(line)
                except json.JSONDecodeError:
                    continue

                if event.get("type") == "chunk":
                    full_text += event.get("content", "")
                elif event.get("type") == "done":
                    if "response" in event:
                        full_text = event["response"]
                    break

            return full_text

        except Exception as e:
            last_err = e
            if debug:
                print(f"Attempt {attempt} failed: {e}")
            time.sleep(backoff_sec * attempt)

    raise RuntimeError(f"TU LLM call failed: {last_err}")

In [24]:
def predict_from_sentence(gml_id, sentence, debug=False):
    gml_id = gml_id.item() if hasattr(gml_id, "item") else gml_id
    sentence = "" if sentence is None else str(sentence)

    try:
        raw = call_tu_llm(sentence, debug=debug)
        obj = extract_first_json_object(raw)
        validate(obj)

        return {
            "gml_id": gml_id,
            "interpreted_type": obj["interpreted_type"],
            "mid_labels": obj["mid_labels"],
            "bosserhof_class": obj["bosserhof_class"],
            # "short_reason": obj["reason"][:100],
            "short_reason": obj["reason"],
        }

    except Exception as e:
        return {
            "gml_id": gml_id,
            "interpreted_type": "error",
            "mid_labels": [],
            "bosserhof_class": None,
            "short_reason": f"Failed: {e}",
        }

In [25]:
def classify_first_n(pois_df: pd.DataFrame, n=50, debug=False) -> pd.DataFrame:
    df = pois_df.head(n).copy()
    results = []

    for i, row in df.iterrows():
        gml_id = row.get("gml_id", row.get("id", i))
        sentence = row.get("sentence", "")

        res = predict_from_sentence(
            gml_id=gml_id,
            sentence=sentence,
            debug=debug,
        )

        results.append(res)
        print(f"Done {len(results)}/{len(df)}")

    return df.reset_index(drop=True).join(
        pd.DataFrame(results), rsuffix="_pred"
    )

In [26]:
import pandas as pd
import time

def rerun_failed_classifications(
    out_df: pd.DataFrame,
    max_rounds: int = 20,
    wait_seconds: int = 2,
    debug: bool = False
) -> pd.DataFrame:
    result_df = out_df.copy().reset_index(drop=True)

    for round_num in range(1, max_rounds + 1):
        invalid_mask = (
            result_df["interpreted_type"].isna()
            | (result_df["interpreted_type"].astype(str).str.strip().str.lower() == "error")
            | (result_df["interpreted_type"].astype(str).str.strip() == "")
        )

        n_invalid = invalid_mask.sum()
        print(f"Round {round_num}: {n_invalid} invalid rows remaining")

        if n_invalid == 0:
            print("All rows classified successfully.")
            break

        failed_rows = result_df.loc[invalid_mask, ["gml_id", "sentence"]].copy()
        rerun_df = classify_first_n(failed_rows, n=len(failed_rows), debug=debug)

        result_df = result_df.set_index("gml_id")
        rerun_df = rerun_df.set_index("gml_id")

        for col in rerun_df.columns:
            if col != "sentence":
                result_df.loc[rerun_df.index, col] = rerun_df[col]

        result_df = result_df.reset_index()
        time.sleep(wait_seconds)

    return result_df


sample_df = pois.sample(n=68)
out5 = classify_first_n(sample_df, n=68, debug=False)
out5 = rerun_failed_classifications(out5, max_rounds=20, wait_seconds=2, debug=False)

Done 1/68
Done 2/68
Done 3/68
Done 4/68
Done 5/68
Done 6/68
Done 7/68
Done 8/68
Done 9/68
Done 10/68
Done 11/68
Done 12/68
Done 13/68
Done 14/68
Done 15/68
Done 16/68
Done 17/68
Done 18/68
Done 19/68
Done 20/68
Done 21/68
Done 22/68
Done 23/68
Done 24/68
Done 25/68
Done 26/68
Done 27/68
Done 28/68
Done 29/68
Done 30/68
Done 31/68
Done 32/68
Done 33/68
Done 34/68
Done 35/68
Done 36/68
Done 37/68
Done 38/68
Done 39/68
Done 40/68
Done 41/68
Done 42/68
Done 43/68
Done 44/68
Done 45/68
Done 46/68
Done 47/68
Done 48/68
Done 49/68
Done 50/68
Done 51/68
Done 52/68
Done 53/68
Done 54/68
Done 55/68
Done 56/68
Done 57/68
Done 58/68
Done 59/68
Done 60/68
Done 61/68
Done 62/68
Done 63/68
Done 64/68
Done 65/68
Done 66/68
Done 67/68
Done 68/68
Round 1: 2 invalid rows remaining
Done 1/2
Done 2/2
Round 2: 1 invalid rows remaining
Done 1/1
Round 3: 1 invalid rows remaining
Done 1/1
Round 4: 0 invalid rows remaining
All rows classified successfully.


In [27]:
out5.head()

,gml_id,osm_names,volume_m3,sentence,gml_id_pred,interpreted_type,mid_labels,bosserhof_class,short_reason
0,17369,None,69.805806,This row describes one building. Estimated bui...,17369,Unknown building,[],None,The only information provided is the building’...
1,401601,None,159.501822,This row describes one building. Estimated bui...,401601,undetermined building,[],None,The text only states that this row describes a...
2,23794,None,103.758903,This row describes one building. Estimated bui...,23794,Unknown building,[],None,The input only provides a building volume esti...
3,400209,None,9477.983002,This row describes one building. Estimated bui...,400209,Unknown building,[],None,The input provides only a generic reference to...
4,38102,None,212.246206,This row describes one building. Estimated bui...,38102,Unspecified building (unknown use),[],None,The input only provides an estimated building ...


In [28]:
from pathlib import Path
import geopandas as gpd

gdf_geom = gpd.read_file(
    r"Areas-of-interest-POIs\condensed_buildings_with_pois.gpkg"
)

gdf_geom["gml_id"] = gdf_geom["gml_id"].astype(str)
out5["gml_id"] = out5["gml_id"].astype(str)

geom_df = gdf_geom[["gml_id", "geometry"]].copy()

merged_df = out5.merge(geom_df, on="gml_id", how="left")

final_gdf = gpd.GeoDataFrame(
    merged_df,
    geometry="geometry",
    crs=gdf_geom.crs
)

out_path = Path(
    r"Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-9.gpkg"
)

out_path.parent.mkdir(parents=True, exist_ok=True)

final_gdf.to_file(out_path, driver="GPKG")

print("Saved:", out_path.exists())
print(out_path.resolve())

Saved: True
C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline\Areas-of-interest-POIs\Data-created(2nd LLM run)\final_predictions_with_geometry_rerun-9.gpkg
